# V2 Model Evaluation
*Experimental engineering walkthrough*

## 01 — Why this problem matters

Your college approves a scholarship using your enrollment proof and income certificate. Your library membership is approved separately.

Later, the income certificate is corrected. **The scholarship approval needs rechecking; the library membership should keep working.**

My project tracks which evidence supports each result, so changed evidence triggers review of the affected work.

### Setup
Open from the repository root, or set `CERT_RECOVERY_REPO`. Use the existing project/test environment with Jupyter.

For optional model replay, use the existing model-environment kernel and set `CERT_RECOVERY_V2_RUN` to the private V2 run. This notebook does not train or download models.

In [1]:
import os
import sys
import json
import math
import tempfile
import importlib.util
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import Mock, patch
from IPython.display import display, Markdown

repo_location = os.environ.get("CERT_RECOVERY_REPO", "")
v2_location = os.environ.get("CERT_RECOVERY_V2_RUN", "")
candidates = [Path(repo_location).expanduser()] if repo_location else [
    Path.cwd(), *Path.cwd().parents, Path.cwd() / "cap-stone-current"]
root = next((p.resolve() for p in candidates
             if (p / "src/cert_recovery/controlled_study.py").is_file()), None)
if root is None:
    raise RuntimeError("Set CERT_RECOVERY_REPO to the current checkout.")
sys.path.insert(0, str(root / "src"))

from cert_recovery.data_pipeline import clean_rows, load_verified_splits
from cert_recovery.model_pipeline import _infer, _prediction_report
from cert_recovery.controlled_study import verify_frozen, execute_stage
from cert_recovery.pinned_smoke import sha256_file
import cert_recovery.controlled_study as study

checks = {}
def expect_error(action, error_type):
    try:
        action()
    except error_type as error:
        print(f"{type(error).__name__}: {error}")
        return str(error)
    raise AssertionError(f"Expected {error_type.__name__}, but execution continued.")

print("V2 functions loaded. Demonstration uses disposable fixtures.")

V2 functions loaded. Demonstration uses disposable fixtures.


---
## 02 — What went wrong before V2
**What are we testing?** Can a confident semantic answer still be wrong?

Read the genuine historical NLI response; no new service request is made.

In [2]:
fixture = root / "tests/remote/fixtures/case_003_recorded_response.json"
record = json.loads(fixture.read_text(encoding="utf-8"))["record"]
answer = record["normalized_output"]

print("Premise:   ", record["input"]["premise"])
print("Hypothesis:", record["input"]["hypothesis"])
print("Expected:  ", record["expected_label"])
print("Actual:    ", answer["label"])
print("Reported confidence:", answer["confidence_text"])
print("Correct:", record["matches_expected_label"])

assert record["expected_label"] == "neutral"
assert answer["label"] == "contradiction"

Premise:    A man is eating pizza.
Hypothesis: The man owns a bicycle.
Expected:   neutral
Actual:     contradiction
Reported confidence: 99.57%
Correct: False


**What did we learn?** Eating pizza says nothing about bicycle ownership, yet the service confidently returned contradiction. This is historical **NLI evidence**, not a binary original-versus-V2 comparison; the service's loaded weights were unknown.

---
## 03 — What V2 is supposed to do
V2 analyzes a complete pair of texts and suggests whether the named source supports the target.

**Input → Verify → Analyze → Predict → Check → Record**

The model's answer is advisory. The independent certificate engine checks its declared rules and current evidence before accepting certificate validity.

### A small decision-rule check
**What are we testing?** The reporting code applies the frozen threshold, rather than assuming 0.50. The probability below is explicitly synthetic.

In [3]:
decision_row = {"pair_id": "threshold-demo", "world_id": "demo",
                "premise": "Q uses A only.", "hypothesis": "Is A a support of Q?",
                "label": 1, "source": "Synthetic decision-rule example"}
decision_report = _prediction_report([decision_row], [0.30], 0.20, 10)
decision = decision_report["predictions"][0]

print("Input: synthetic P(dependency) = 0.30")
print("Threshold:", decision_report["threshold"])
print("Expected: depends_on, because 0.30 >= 0.20")
print("Actual:  ", "depends_on" if decision["prediction"] else "independent")
print("Predicted-label probability:", decision["predicted_label_probability"])
assert decision["prediction"] == 1
assert decision["predicted_label_probability"] == 0.30
expect_error(lambda: _prediction_report([decision_row], [float("nan")], 0.20, 10), ValueError)

Input: synthetic P(dependency) = 0.30
Threshold: 0.2
Expected: depends_on, because 0.30 >= 0.20
Actual:   depends_on
Predicted-label probability: 0.3
ValueError: Probabilities must be finite and in [0,1]
Out[3]: 'Probabilities must be finite and in [0,1]'


**What did we learn?** The decision follows the supplied threshold and rejects an invalid probability. This check does not measure model accuracy or choose a new threshold.

---
## 04 — Can V2 preserve the complete input?
**What are we testing?** Clean whitespace and Unicode without losing the target scope, negation or number.

In [4]:
raw_pair = {"pair_id": "mentor-clean", "world_id": "mentor-world",
            "premise": "  Q-only:\n source Cafe\u0301 is not 17.  All sources have no supports. ",
            "hypothesis": " Q uses Cafe\u0301 only.\tAll other sources are excluded. ",
            "label": 1, "source": "Synthetic input-handling example"}
cleaned, _ = clean_rows([raw_pair])
expected = "Q-only: source Café is not 17. All sources have no supports."

print("Input:   ", repr(raw_pair["premise"]))
print("Expected:", expected)
print("Actual:  ", cleaned[0]["premise"])
print("Hypothesis:", cleaned[0]["hypothesis"])
assert cleaned[0]["premise"] == expected
assert cleaned[0]["hypothesis"] == "Q uses Café only. All other sources are excluded."
checks["Input handling"] = True

Input:    '  Q-only:\n source Café is not 17.  All sources have no supports. '
Expected: Q-only: source Café is not 17. All sources have no supports.
Actual:   Q-only: source Café is not 17. All sources have no supports.
Hypothesis: Q uses Café only. All other sources are excluded.


**What did we learn?** Formatting changed; the important text survived. This is a synthetic input-handling check, not model inference.

---
## 05 — Can we detect changed evidence?
**What are we testing?** Accept a frozen bundle, then reject a changed validation row.

Reuse the existing V2 test fixture: six synthetic rows and real source hashes, with no weights. All edits stay in a temporary directory.

In [5]:
spec = importlib.util.spec_from_file_location("v2_demo_fixture", root / "tests/test_v2_execution.py")
fixture_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fixture_module)
temporary = tempfile.TemporaryDirectory(prefix="v2_notebook_")

def fresh_fixture(name):
    folder = Path(temporary.name) / name
    folder.mkdir()
    return fixture_module.frozen_run.__wrapped__(folder)

integrity_run = fresh_fixture("integrity")
verify_frozen(integrity_run)
print("Untouched bundle: accepted")

validation_path = integrity_run / "splits/validation.jsonl"
original_bytes = validation_path.read_bytes()
rows = [json.loads(line) for line in validation_path.read_text().splitlines()]
rows[0]["premise"] = "Altered demonstration text"
validation_path.write_text("".join(json.dumps(row) + "\n" for row in rows))

print("Input: validation premise changed. Expected: block.")
message = expect_error(lambda: verify_frozen(integrity_run), ValueError)
assert message == "Empty or altered split: validation"

validation_path.write_bytes(original_bytes)
verify_frozen(integrity_run)
print("Original bytes restored: accepted again")
checks["Integrity checks"] = True

Untouched bundle: accepted
Input: validation premise changed. Expected: block.
ValueError: Empty or altered split: validation
Original bytes restored: accepted again


**What did we learn?** V2 detects the altered split. Hash agreement establishes consistency with the frozen evidence, not that its annotations are correct.

---
## 06 — Can we prevent invalid execution order?
**What are we testing?** Request comparison before successful training.

In [6]:
ordering_run = fresh_fixture("ordering")
manifest_before = (ordering_run / "manifest.json").read_bytes()

print("Input: comparison requested before training. Expected: block.")
message = expect_error(lambda: execute_stage(ordering_run, "comparison"), ValueError)
attempt_count = len(list(ordering_run.glob("attempt_*")))
print("Attempts created:", attempt_count)

assert message == "Successful selected training required"
assert attempt_count == 0
assert (ordering_run / "manifest.json").read_bytes() == manifest_before
checks["Execution controls"] = True

Input: comparison requested before training. Expected: block.
ValueError: Successful selected training required
Attempts created: 0


**What did we learn?** Comparison was blocked before any model execution or new attempt.

---
## 07 — Do failures remain visible?
**What are we testing?** Inject a backend failure and check that the real stage controller saves `ERROR` and prevents an overwrite.

Only the external training callback and hardware adapter are simulated; no real training occurs.

In [7]:
failure_run = fresh_fixture("failure")
cpu_adapter = SimpleNamespace(set_num_threads=Mock(), cuda=SimpleNamespace(is_available=lambda: False))

with patch.dict(sys.modules, {"torch": cpu_adapter}), patch.object(
        study, "run_finetuning", side_effect=RuntimeError("Controlled demonstration failure")) as backend:
    expect_error(lambda: execute_stage(failure_run, "training"), RuntimeError)
    assert backend.call_count == 1

state = json.loads((failure_run / "manifest.json").read_text())["stages"]["training"]
saved_state = json.loads((Path(state["attempt"]) / "status.json").read_text())
print("Expected: ERROR retained in both records")
print("Actual status:", state["status"])
print("Saved error:", state["error"])
print("Attempt and manifest agree:", saved_state == state)
assert state["status"] == "ERROR" and saved_state == state

print("Retry the same stage. Expected: reject.")
expect_error(lambda: execute_stage(failure_run, "training"), FileExistsError)
checks["Failure handling"] = True

RuntimeError: Controlled demonstration failure
Expected: ERROR retained in both records
Actual status: ERROR
Saved error: {'type': 'RuntimeError', 'message': 'Controlled demonstration failure'}
Attempt and manifest agree: True
Retry the same stage. Expected: reject.
FileExistsError: Stage already attempted; preserve attempt and choose a separately authorized new run


**What did we learn?** The failure stays visible; the same attempt cannot silently be retried or overwritten.

### What about invalid hardware?
Simulate a GPU being available for this frozen **CPU-only** experiment. The guard should stop before the training callback.

In [8]:
hardware_run = fresh_fixture("hardware")
gpu_adapter = SimpleNamespace(set_num_threads=Mock(), cuda=SimpleNamespace(is_available=lambda: True))

with patch.dict(sys.modules, {"torch": gpu_adapter}), patch.object(study, "run_finetuning") as backend:
    message = expect_error(lambda: execute_stage(hardware_run, "training"), ValueError)
    print("Training callback called:", backend.called)
    assert not backend.called

hardware_state = json.loads((hardware_run / "manifest.json").read_text())["stages"]["training"]
assert message == "This frozen experiment is local CPU only"
assert hardware_state["status"] == "ERROR"
print("Persisted status:", hardware_state["status"])
checks["Hardware guard"] = True

ValueError: This frozen experiment is local CPU only
Training callback called: False
Persisted status: ERROR


**What did we learn?** The protocol-specific hardware guard blocked execution and recorded the error. DeBERTa itself is not restricted to CPU.

---
## 08 — Real V2 model replay
**What are we testing?** Replay up to three verified validation examples using the existing private checkpoint and frozen threshold. Compare annotations, saved predictions and current predictions.

Select a correct dependent case, a correct independent case and a high-confidence error where available. These are qualitative examples, not an unbiased accuracy sample.

In [9]:
replay_status = "NOT RUN"
replay_report = None
replay_rows = []
saved_by_id = {}
selection = None
replay_error = None

if not v2_location:
    print("NOT RUN — private V2 checkpoint unavailable.")
else:
    try:
        private_run = Path(v2_location).expanduser().resolve()
        replay_config, _ = verify_frozen(private_run)
        selection = json.loads((private_run / "checkpoints/selection.json").read_text())
        checkpoint = private_run / "checkpoints/best"
        if sha256_file(checkpoint / "model.safetensors") != selection["checkpoint_sha256"]:
            raise ValueError("V2 checkpoint hash mismatch")
        for filename, digest in selection["checkpoint_file_hashes"].items():
            if sha256_file(checkpoint / filename) != digest:
                raise ValueError(f"Checkpoint file changed: {filename}")

        splits, _ = load_verified_splits(replay_config)
        saved = json.loads((private_run / "comparisons/validation_candidate_v2.json").read_text())
        saved_by_id = {row["pair_id"]: row for row in saved["predictions"]}
        for row in splits["validation"]:
            reference = saved_by_id[row["pair_id"]]
            if any(reference[key] != row[key] for key in ("premise", "hypothesis", "label")):
                raise ValueError("Saved output differs from verified validation input/label")

        selected_ids = []
        for label in (1, 0):
            eligible = [row for row in saved["predictions"] if row["label"] == label == row["prediction"]]
            if eligible:
                selected_ids.append(eligible[0]["pair_id"])
        errors = [row for row in saved["predictions"] if row["prediction"] != row["label"]]
        if errors:
            worst = max(errors, key=lambda row: row["probability"] if row["prediction"] else 1-row["probability"])
            selected_ids.append(worst["pair_id"])
        for row in splits["validation"]:
            if len(selected_ids) >= 3:
                break
            if row["pair_id"] not in selected_ids:
                selected_ids.append(row["pair_id"])
        verified_rows = {row["pair_id"]: row for row in splits["validation"]}
        replay_rows = [verified_rows[pair_id] for pair_id in selected_ids]

        os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True, trust_remote_code=False)
        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint, local_files_only=True, trust_remote_code=False, use_safetensors=True)
        if model.config.id2label != {0: "independent", 1: "depends_on"}:
            raise ValueError("Unexpected V2 binary label mapping")
        if any(len(tokenizer(row["premise"], row["hypothesis"], truncation=False)["input_ids"])
               > replay_config["model"]["max_length"] for row in replay_rows):
            raise ValueError("Replay would truncate the task context")

        raw_outputs = []
        with patch.object(torch.cuda, "is_available", return_value=False):
            probabilities = _infer(replay_config, replay_rows, model, tokenizer, 1, raw_outputs)
        replay_report = _prediction_report(replay_rows, probabilities, selection["threshold"],
                                            replay_config["model"]["calibration_bins"], raw_outputs)
        replay_status = "EXECUTED"
        print("Real checkpoint replay executed on CPU.")
        print("Model:", replay_config["model"]["pretrained_id"])
        print("Frozen threshold:", selection["threshold"])
        del model, tokenizer
    except Exception as error:
        replay_status = "ERROR"
        replay_error = f"{type(error).__name__}: {error}"
        print("ERROR — replay stopped:", replay_error)

NOT RUN — private V2 checkpoint unavailable.


In [10]:
if replay_report is not None:
    for actual in replay_report["predictions"]:
        saved_row = saved_by_id[actual["pair_id"]]
        label_name = lambda value: "depends_on" if value else "independent"
        print("Pair:", actual["pair_id"])
        print("Premise:", actual["premise"])
        print("Hypothesis:", actual["hypothesis"])
        print("Expected annotation:", label_name(actual["label"]))
        print("Saved V2 prediction:", label_name(saved_row["prediction"]))
        print("Actual V2 prediction:", label_name(actual["prediction"]))
        print("Dependency probability:", f"{actual['probability']:.2%}")
        print("Matches annotation:", actual["prediction"] == actual["label"])
        print("Matches saved prediction:", actual["prediction"] == saved_row["prediction"])
        print("Probability replay within 1e-5:", math.isclose(
            actual["probability"], saved_row["probability"], abs_tol=1e-5, rel_tol=1e-5))
        print()
else:
    print("No fresh model predictions to compare.")

No fresh model predictions to compare.


**What did we learn?** The status above tells us whether trained-model behavior was observed. Matching saved predictions checks reproducibility; matching provisional annotations does not establish generalization.

Raw logits, full probabilities and the frozen configuration remain available in `replay_report`, `selection` and `replay_config` when replay succeeds.

### Optional original-versus-V2 comparison
Read saved original predictions only when the same verified input and label are available. This is not fresh original-model inference.

In [11]:
if replay_report is not None:
    original_path = private_run / "comparisons/validation_original.json"
    if original_path.is_file():
        original_report = json.loads(original_path.read_text())
        original_by_id = {row["pair_id"]: row for row in original_report["predictions"]}
        for actual in replay_report["predictions"]:
            before = original_by_id[actual["pair_id"]]
            if any(before[key] != actual[key] for key in ("premise", "hypothesis", "label")):
                raise ValueError("Original comparator input/label mismatch")
            print(actual["pair_id"], "| expected:", actual["label"],
                  "| original:", before["prediction"], "| V2:", actual["prediction"])
    else:
        print("Saved original comparator unavailable.")
else:
    print("No original-versus-V2 comparison was measured in this execution.")

No original-versus-V2 comparison was measured in this execution.


---
## 09 — What did we actually prove?
Utility checks demonstrate the observed safeguards. They do not prove predictive accuracy.

In [12]:
summary = ["| Item | Observed status |", "|---|---|"]
for name in ("Input handling", "Integrity checks", "Execution controls", "Failure handling", "Hardware guard"):
    summary.append(f"| {name} | {'Passed' if checks.get(name) else 'Not verified'} |")
summary.append(f"| Real V2 replay | {replay_status} |")
summary.append("| Measured model improvement | Not claimed |")
display(Markdown("\n".join(summary)))

| Item | Observed status |
|---|---|
| Input handling | Passed |
| Integrity checks | Passed |
| Execution controls | Passed |
| Failure handling | Passed |
| Hardware guard | Passed |
| Real V2 replay | NOT RUN |
| Measured model improvement | Not claimed |

---
## 10 — What still needs testing?
- Replay the existing private checkpoint and inspect its actual mistakes.
- Review labels against **declared support membership**: unchanged output values do not prove independence.
- Test valid renames, role changes and reviewed domain examples before claiming broader accuracy.

V3 remains outside this demonstration.

In [13]:
temporary.cleanup()
print("Disposable fixtures removed. Repository code and private run artifacts were not changed.")

Disposable fixtures removed. Repository code and private run artifacts were not changed.
